In [1]:
# calculation cell for esm-up2p0 is in Oxygen_TEST.ipynb

In [2]:
import xarray as xr
import numpy as np
import time
import os
import glob
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# Settings
# ============================================================

EXPERIMENTS = {
    "esm-piControl": {
        "o2_dir": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest",
        "mld_dir": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/mlotst/gn/latest",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/o2-inventories",
        "exp_id": "esm-piControl",
        "grid": "gr",
    },

    "esm-up2p0-swl2p0": {
        "o2_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "mld_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/o2-inventories",
        "exp_id": "esm-up2p0-swl2p0",
        "grid": "gr",
    },

    "esm-up2p0-swl4p0": {
        "o2_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "mld_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/o2-inventories",
        "exp_id": "esm-up2p0-swl4p0",
        "grid": "gr",
    },
}


TARGET_DEPTHS = [100, 200, 300, 500, 700, 1000, 1500, 2000]


# ============================================================
# Helper functions
# ============================================================

def find_time_suffix(filename):
    return os.path.basename(filename).split("_")[-1]


def find_matching_mld_file(mld_dir, exp_id, grid, time_suffix):
    pattern = os.path.join(
        mld_dir,
        f"mlotst_Omon_NorESM2-LM_{exp_id}_r1i1p1f1_gn_{time_suffix}"
    )

    files = sorted(glob.glob(pattern))

    if len(files) == 0:
        return None

    return files[0]


def get_lev_bounds(ds):
    if "lev_bnds" in ds:
        lev_bnds = ds["lev_bnds"]
    elif "lev_bounds" in ds:
        lev_bnds = ds["lev_bounds"]
    else:
        raise KeyError("No vertical bounds variable found: expected lev_bnds or lev_bounds")

    bnd_dim = [d for d in lev_bnds.dims if d not in ["lev"]][0]

    z_top = lev_bnds.isel({bnd_dim: 0})
    z_bot = lev_bnds.isel({bnd_dim: 1})

    return z_top, z_bot


def align_mld_to_o2_grid(mlotst, o2):
    """
    Align MLD to the O2 horizontal grid.

    In NorESM2-LM files used here, o2 has grid_label='gr'
    because it is interpolated to z-levels, but its horizontal
    grid is still the same tripolar j/i grid as mlotst.
    Therefore, no horizontal remapping is needed when j/i sizes match.
    """

    target = o2.isel(time=0, lev=0)

    if ("j" in mlotst.dims) and ("i" in mlotst.dims):
        if (mlotst.sizes["j"] == target.sizes["j"]) and (mlotst.sizes["i"] == target.sizes["i"]):
            return mlotst

    raise ValueError("MLD and O2 horizontal grids do not match. Check j/i dimensions.")


def process_one_file(o2_file, exp_name, exp_info, output_dir):
    loop_start_time = time.time()

    exp_id = exp_info["exp_id"]
    grid = exp_info["grid"]
    time_suffix = find_time_suffix(o2_file)

    mld_file = find_matching_mld_file(
        mld_dir=exp_info["mld_dir"],
        exp_id=exp_id,
        grid=grid,
        time_suffix=time_suffix,
    )

    output_filename = (
        f"o2-inventories_Omon_NorESM2-LM_"
        f"{exp_id}_r1i1p1f1_{grid}_{time_suffix}"
    )
    output_path = os.path.join(output_dir, output_filename)

    print(f"  Processing period: {time_suffix.replace('.nc', '')}")

    if mld_file is None:
        print("    -> WARNING: Matching MLD file not found. Skipping.")
        return

    if os.path.exists(output_path):
        print("    -> Output already exists. Skipping.")
        return

    ds_o2 = xr.open_dataset(o2_file, use_cftime=True)
    ds_mld = xr.open_dataset(mld_file, use_cftime=True)

    o2 = ds_o2["o2"]
    mld = ds_mld["mlotst"]

    mld_aligned = align_mld_to_o2_grid(mld, o2)

    z_top, z_bot = get_lev_bounds(ds_o2)

    out_vars = {}

    out_vars["mlotst"] = mld_aligned
    out_vars["mlotst"].attrs["long_name"] = "Ocean Mixed Layer Thickness on O2 grid"
    out_vars["mlotst"].attrs["units"] = "m"

    # Mixed-layer O2 inventory
    overlap_mld = np.minimum(z_bot, mld_aligned) - z_top
    overlap_mld = xr.where(overlap_mld > 0, overlap_mld, 0)

    o2_inv_mld = (o2 * overlap_mld).sum(dim="lev")
    o2_inv_mld.name = "o2_inv_mld"
    o2_inv_mld.attrs["long_name"] = "O2 inventory above mixed layer depth"
    o2_inv_mld.attrs["units"] = "mol m-2"

    out_vars["o2_inv_mld"] = o2_inv_mld

    # Fixed-depth O2 inventories
    for depth in TARGET_DEPTHS:
        overlap_fixed = np.minimum(z_bot, depth) - z_top
        overlap_fixed = xr.where(overlap_fixed > 0, overlap_fixed, 0)

        o2_inv_fixed = (o2 * overlap_fixed).sum(dim="lev")
        var_name = f"o2_inv_{depth}m"

        o2_inv_fixed.name = var_name
        o2_inv_fixed.attrs["long_name"] = f"O2 inventory from surface to {depth} m"
        o2_inv_fixed.attrs["units"] = "mol m-2"

        out_vars[var_name] = o2_inv_fixed

    # Total water-column O2 inventory
    overlap_total = z_bot - z_top
    overlap_total = xr.where(overlap_total > 0, overlap_total, 0)

    o2_inv_total = (o2 * overlap_total).sum(dim="lev")
    o2_inv_total.name = "o2_inv_total"
    o2_inv_total.attrs["long_name"] = "Total water-column O2 inventory"
    o2_inv_total.attrs["units"] = "mol m-2"

    out_vars["o2_inv_total"] = o2_inv_total

    # Build output dataset
    out_ds = xr.Dataset(out_vars)

    # Add useful coordinates if available
    for coord_name in ["latitude", "longitude", "lat", "lon"]:
        if coord_name in ds_o2:
            out_ds = out_ds.assign_coords({coord_name: ds_o2[coord_name]})

    out_ds.attrs["source_o2_file"] = o2_file
    out_ds.attrs["source_mld_file"] = mld_file
    out_ds.attrs["description"] = "O2 inventories calculated from monthly O2 and MLD fields"

    # Compression
    encoding = {
        var: {"zlib": True, "complevel": 4}
        for var in out_ds.data_vars
    }

    out_ds.to_netcdf(output_path, encoding=encoding)

    out_ds.close()
    ds_o2.close()
    ds_mld.close()

    print(f"    -> Saved {output_filename}")
    print(f"    -> Took {time.time() - loop_start_time:.2f} sec")


# ============================================================
# Main loop
# ============================================================

total_start_time = time.time()

print("Starting production pipeline: O2 inventories for multiple experiments")
print("=" * 80)

for exp_name, exp_info in EXPERIMENTS.items():

    print("\n" + "=" * 80)
    print(f"Experiment: {exp_name}")
    print("=" * 80)

    output_dir = exp_info["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    o2_pattern = os.path.join(
        exp_info["o2_dir"],
        f"o2_Omon_NorESM2-LM_{exp_info['exp_id']}_r1i1p1f1_{exp_info['grid']}_*.nc"
    )

    o2_files = sorted(glob.glob(o2_pattern))

    print(f"Found {len(o2_files)} O2 files.")

    if len(o2_files) == 0:
        print("No O2 files found. Check path/grid name.")
        continue

    print("First file:", o2_files[0])
    print("Last file :", o2_files[-1])

    for idx, o2_file in enumerate(o2_files, 1):
        print(f"\n[{idx}/{len(o2_files)}]")
        process_one_file(
            o2_file=o2_file,
            exp_name=exp_name,
            exp_info=exp_info,
            output_dir=output_dir,
        )

print("\n" + "=" * 80)
print("All processing complete.")
print(f"Total elapsed time: {(time.time() - total_start_time) / 60:.2f} minutes")

Starting production pipeline: O2 inventories for multiple experiments

Experiment: esm-piControl
Found 25 O2 files.
First file: /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Last file : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_209101-210012.nc

[1/25]
  Processing period: 185101-186012
    -> Output already exists. Skipping.

[2/25]
  Processing period: 186101-187012
    -> Output already exists. Skipping.

[3/25]
  Processing period: 187101-188012
    -> Output already exists. Skipping.

[4/25]
  Processing period: 188101-189012
    -> Output already exists. Skipping.

[5/25]
  Processing period: 189101-190012
    -> Output already exists. Skipping.

[6/25]
  Processing period: 190101-191012
    -> Output already exists. Skipping.

[7/25]
  Processing period: 191101-192012
    -> Output a

In [2]:
import xarray as xr
import numpy as np
import time
import os
import glob
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# Settings
# ============================================================

EXPERIMENTS = {
    "esm-piControl": {
        "o2_dir": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/o2-inventories-mesopelagic",
        "exp_id": "esm-piControl",
        "grid": "gr",
    },

    "esm-up2p0-swl2p0": {
        "o2_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/o2-inventories-mesopelagic",
        "exp_id": "esm-up2p0-swl2p0",
        "grid": "gr",
    },

    "esm-up2p0-swl4p0": {
        "o2_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/o2-inventories-mesopelagic",
        "exp_id": "esm-up2p0-swl4p0",
        "grid": "gr",
    },
    "esm-up2p0": {
        "o2_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010",
        "output_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/o2-inventories-mesopelagic",
        "exp_id": "esm-up2p0",
        "grid": "gr",
    },
}

Z_MIN = 200.0
Z_MAX = 1000.0


# ============================================================
# Helper functions
# ============================================================

def find_time_suffix(filename):
    return os.path.basename(filename).split("_")[-1]


def get_lev_bounds(ds):
    if "lev_bnds" in ds:
        lev_bnds = ds["lev_bnds"]
    elif "lev_bounds" in ds:
        lev_bnds = ds["lev_bounds"]
    else:
        raise KeyError("No vertical bounds variable found: expected lev_bnds or lev_bounds")

    bnd_dim = [d for d in lev_bnds.dims if d not in ["lev"]][0]

    z_top = lev_bnds.isel({bnd_dim: 0})
    z_bot = lev_bnds.isel({bnd_dim: 1})

    return z_top, z_bot


def calculate_layer_overlap(z_top, z_bot, z_min, z_max):
    """
    Calculate vertical overlap thickness between model layers and target depth range.

    z_top, z_bot: layer bounds [m], positive downward
    z_min, z_max: target range [m], positive downward
    """

    overlap = np.minimum(z_bot, z_max) - np.maximum(z_top, z_min)
    overlap = xr.where(overlap > 0, overlap, 0)

    return overlap


def process_one_file(o2_file, exp_name, exp_info, output_dir):
    loop_start_time = time.time()

    exp_id = exp_info["exp_id"]
    grid = exp_info["grid"]
    time_suffix = find_time_suffix(o2_file)

    output_filename = (
        f"o2-inv-200-1000m_Omon_NorESM2-LM_"
        f"{exp_id}_r1i1p1f1_{grid}_{time_suffix}"
    )
    output_path = os.path.join(output_dir, output_filename)

    print(f"  Processing period: {time_suffix.replace('.nc', '')}")

    if os.path.exists(output_path):
        print("    -> Output already exists. Skipping.")
        return

    ds_o2 = xr.open_dataset(o2_file, use_cftime=True)

    o2 = ds_o2["o2"]
    z_top, z_bot = get_lev_bounds(ds_o2)

    overlap_200_1000 = calculate_layer_overlap(
        z_top=z_top,
        z_bot=z_bot,
        z_min=Z_MIN,
        z_max=Z_MAX,
    )

    o2_inv_200_1000m = (o2 * overlap_200_1000).sum(dim="lev")
    o2_inv_200_1000m.name = "o2_inv_200_1000m"
    o2_inv_200_1000m.attrs["long_name"] = "O2 inventory from 200 m to 1000 m"
    o2_inv_200_1000m.attrs["units"] = "mol m-2"
    o2_inv_200_1000m.attrs["depth_range"] = "200-1000 m"
    o2_inv_200_1000m.attrs["note"] = (
        "Calculated by vertically integrating o2 over partial-cell overlap "
        "between layer bounds and 200-1000 m."
    )

    out_ds = xr.Dataset({"o2_inv_200_1000m": o2_inv_200_1000m})

    for coord_name in ["latitude", "longitude", "lat", "lon"]:
        if coord_name in ds_o2:
            out_ds = out_ds.assign_coords({coord_name: ds_o2[coord_name]})

    out_ds.attrs["source_o2_file"] = o2_file
    out_ds.attrs["description"] = "Mesopelagic O2 inventory calculated from monthly O2 fields"
    out_ds.attrs["depth_min_m"] = Z_MIN
    out_ds.attrs["depth_max_m"] = Z_MAX

    encoding = {
        var: {"zlib": True, "complevel": 4}
        for var in out_ds.data_vars
    }

    out_ds.to_netcdf(output_path, encoding=encoding)

    out_ds.close()
    ds_o2.close()

    print(f"    -> Saved {output_filename}")
    print(f"    -> Took {time.time() - loop_start_time:.2f} sec")


# ============================================================
# Main loop
# ============================================================

total_start_time = time.time()

print("Starting production pipeline: O2 inventory from 200 to 1000 m")
print("=" * 80)

for exp_name, exp_info in EXPERIMENTS.items():

    print("\n" + "=" * 80)
    print(f"Experiment: {exp_name}")
    print("=" * 80)

    output_dir = exp_info["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    o2_pattern = os.path.join(
        exp_info["o2_dir"],
        f"o2_Omon_NorESM2-LM_{exp_info['exp_id']}_r1i1p1f1_{exp_info['grid']}_*.nc"
    )

    o2_files = sorted(glob.glob(o2_pattern))

    print(f"Found {len(o2_files)} O2 files.")

    if len(o2_files) == 0:
        print("No O2 files found. Check path/grid name.")
        continue

    print("First file:", o2_files[0])
    print("Last file :", o2_files[-1])

    for idx, o2_file in enumerate(o2_files, 1):
        print(f"\n[{idx}/{len(o2_files)}]")
        process_one_file(
            o2_file=o2_file,
            exp_name=exp_name,
            exp_info=exp_info,
            output_dir=output_dir,
        )

print("\n" + "=" * 80)
print("All processing complete.")
print(f"Total elapsed time: {(time.time() - total_start_time) / 60:.2f} minutes")

Starting production pipeline: O2 inventory from 200 to 1000 m

Experiment: esm-piControl
Found 25 O2 files.
First file: /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Last file : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_209101-210012.nc

[1/25]
  Processing period: 185101-186012
    -> Output already exists. Skipping.

[2/25]
  Processing period: 186101-187012
    -> Output already exists. Skipping.

[3/25]
  Processing period: 187101-188012
    -> Output already exists. Skipping.

[4/25]
  Processing period: 188101-189012
    -> Output already exists. Skipping.

[5/25]
  Processing period: 189101-190012
    -> Output already exists. Skipping.

[6/25]
  Processing period: 190101-191012
    -> Output already exists. Skipping.

[7/25]
  Processing period: 191101-192012
    -> Output already e